# Weather Scraper

In [2]:
import requests
import pandas as pd
import json

### Testing GraphQL Query

`
curl --request POST \
  --header 'content-type: application/json' \
  --url 'https://api.openbeta.io/' \
  --data '{"query":"query { __typename }"}'
`

In [1]:
url = "https://stg-api.openbeta.io/"

headers = {
    'apollo-require-preflight': 'true',
    "Content-Type": "application/json"
    }

payload = {"query":"query { __typename }"}

response = requests.post(url, json=payload, headers=headers)

if response.status_code == 200:
    print("Success!")
    print(response.json())
else:
    print(f"Error {response.status_code}")
    print(response.text)

NameError: name 'requests' is not defined

In [4]:
import requests
import json

url = "https://api.openbeta.io/"

query = """
{
  areas(filter: {area_name: {match: "Bishop Mountain"} }, limit: 500) {
    areaName
    uuid
    metadata {
      isBoulder
      lat
      lng
      isDestination
    }
    climbs {
      name
      type {
        bouldering
      }
      grades {
        vscale
      }
    }
  }
}
"""

response = requests.post(
    url,
    headers={"content-type": "application/json"},
    json={"query": query}
)

data = response.json()

areas = data["data"]["areas"]
print(f"Number of areas returned: {len(areas)}")
print(json.dumps(areas[0], indent=2))

Number of areas returned: 1
{
  "areaName": "Bishop Mountain",
  "uuid": "29319b4f-f1cf-5381-b8c8-b629eeae1c2f",
  "metadata": {
    "isBoulder": false,
    "lat": 34.447509814814815,
    "lng": -86.38278444444445,
    "isDestination": false
  },
  "climbs": []
}


#### Working query 1, does not get all of Colorado 

In [ ]:
import requests
import json

url = "https://api.openbeta.io/"

query = """
query MyQuery {
  areas(filter: {area_name: {match: "Colorado" } }) {
    areaName
    children {
      areaName
      children {
        areaName
        climbs {
          name
          grades { vscale font }
          type { bouldering }
        }
        metadata { isBoulder }
      }
      climbs {
        name
        grades { vscale }
        type { bouldering }
      }
      metadata { isBoulder }
    }
    climbs {
      name
      grades { vscale }
      type { bouldering }
    }
    metadata { isBoulder }
  }
}
"""


response = requests.post(
    url,
    headers={"content-type": "application/json"},
    json={"query": query}
)

data = response.json()

areas = data["data"]["areas"]
print(f"Number of areas returned: {len(areas)}")
print(json.dumps(areas[0], indent=2))

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

#### Attempting sub queries

In [ ]:
import requests
import json

url = "https://api.openbeta.io/"

query = """
query MyQuery {
  areas(filter: {area_name: {match: "Colorado" } }) {
    areaName
    uuid
    children {
      areaName
      uuid
    }
  }
}
"""


response = requests.post(
    url,
    headers={"content-type": "application/json"},
    json={"query": query}
)

data = response.json()

areas = data["data"]["areas"]
print(f"Number of areas returned: {len(areas)}")
print(json.dumps(areas[0], indent=2))

#### The rest

In [ ]:
def extract_climbs(area, rows, parent_path=""):
    """
    Recursively walks an area and all its nested children,
    pulling out every climb it finds along the way.
    """
    area_name = area.get("areaName", "Unknown")
    full_path = f"{parent_path} > {area_name}" if parent_path else area_name
    
    metadata = area.get("metadata") or {}
    is_boulder_area = metadata.get("isBoulder")
    
    # Grab climbs at THIS level
    climbs = area.get("climbs") or []
    for climb in climbs:
        grades = climb.get("grades") or {}
        climb_type = climb.get("type") or {}
        
        rows.append({
            "area_path": full_path,
            "area_name": area_name,
            "is_boulder_area": is_boulder_area,
            "climb_name": climb.get("name"),
            "grade_v": grades.get("vscale"),
            "grade_font": grades.get("font"),
            "is_bouldering": climb_type.get("bouldering")
        })
    
    # Recurse into children, if they exist
    children = area.get("children") or []
    for child in children:
        extract_climbs(child, rows, parent_path=full_path)


# Run it
rows = []
for area in areas:
    extract_climbs(area, rows)

df = pd.DataFrame(rows)
print(df.shape)
print(df.head(10))

boulder_df = df[df["is_bouldering"] == True].copy()
print(f"\nBouldering climbs found: {len(boulder_df)}")

(2885, 7)
                              area_path         area_name  is_boulder_area  \
0     Colorado City Area > Rock Worship      Rock Worship            False   
1     Colorado City Area > Rock Worship      Rock Worship            False   
2     Colorado City Area > Rock Worship      Rock Worship            False   
3     Colorado City Area > Rock Worship      Rock Worship            False   
4     Colorado City Area > Rock Worship      Rock Worship            False   
5  Colorado City Area > Cottonwood Mesa   Cottonwood Mesa            False   
6  Colorado City Area > Cottonwood Mesa   Cottonwood Mesa            False   
7  Colorado City Area > Cottonwood Mesa   Cottonwood Mesa            False   
8                      Colorado Boulder  Colorado Boulder            False   
9                      Colorado Boulder  Colorado Boulder            False   

                 climb_name grade_v grade_font is_bouldering  
0  Apostacy of Warren Jeffs     NaN        NaN          None  
1    

In [ ]:
print(df.columns.tolist())
print(df.shape)
print(df.head())

['area_path', 'area_name', 'is_boulder_area', 'climb_name', 'grade_v', 'grade_font', 'is_bouldering']
(2885, 7)
                           area_path     area_name  is_boulder_area  \
0  Colorado City Area > Rock Worship  Rock Worship            False   
1  Colorado City Area > Rock Worship  Rock Worship            False   
2  Colorado City Area > Rock Worship  Rock Worship            False   
3  Colorado City Area > Rock Worship  Rock Worship            False   
4  Colorado City Area > Rock Worship  Rock Worship            False   

                 climb_name grade_v grade_font is_bouldering  
0  Apostacy of Warren Jeffs     NaN        NaN          None  
1        Alt Tower Traverse     NaN        NaN          None  
2           Tweeties Demise     NaN        NaN          None  
3                 C-Section     NaN        NaN          None  
4             Polydigitorum     NaN        NaN          None  


In [ ]:
print(boulder_df.columns.tolist())
print(boulder_df.shape)
print(boulder_df.head())

['area_path', 'area_name', 'is_boulder_area', 'climb_name', 'grade_v', 'grade_font', 'is_bouldering']
(679, 7)
                                             area_path  \
8                                     Colorado Boulder   
9                                     Colorado Boulder   
10                                    Colorado Boulder   
45   Colorado > 10 Mile Canyon > Sunshine Buttress ...   
117       Colorado > Breckenridge > Raspberry Boulders   

                                area_name  is_boulder_area  \
8                        Colorado Boulder            False   
9                        Colorado Boulder            False   
10                       Colorado Boulder            False   
45   Sunshine Buttress (aka Wichita Wall)            False   
117                    Raspberry Boulders            False   

               climb_name grade_v grade_font is_bouldering  
8                Freebird      V2        NaN          True  
9                  Pan Am      V2        NaN 

In [ ]:
df.head()

,area_path,area_name,is_boulder_area,climb_name,grade_v,grade_font,is_bouldering
0,Colorado City Area > Rock Worship,Rock Worship,False,Apostacy of Warren Jeffs,NaN,NaN,None
1,Colorado City Area > Rock Worship,Rock Worship,False,Alt Tower Traverse,NaN,NaN,None
2,Colorado City Area > Rock Worship,Rock Worship,False,Tweeties Demise,NaN,NaN,None
3,Colorado City Area > Rock Worship,Rock Worship,False,C-Section,NaN,NaN,None
4,Colorado City Area > Rock Worship,Rock Worship,False,Polydigitorum,NaN,NaN,None


In [ ]:
df[(df["is_bouldering"] == True) & (df["grade_v"] == "V12")].head(10)

,area_path,area_name,is_boulder_area,climb_name,grade_v,grade_font,is_bouldering
132,"Colorado > Black Hawk > Ameristar, The","Ameristar, The",False,Russian Roulette,V12,8A+,True
2061,Colorado > Estes Park Valley > Elkland,Elkland,False,Africa Baambaata,V12,8A+,True


In [ ]:
df["grade_v"].unique()

<StringArray>
[     nan,     'V2',     'V4',   'V3-4',     'V1', 'V-easy',    'V1+',
     'V3',     'V0',   'V0-1',    'V7-', 'V11-12',    'V10',     'V9',
    'V12',     'V5',    'V3+',    'V2+',   'V1-2',     'V6',    'V0-',
    'V4-',   'V5-6',    'V4+',     'V7', 'V10-11',   'V6-7',   'V4-5',
   'V8-9',   'V7-8',     'V8',    'V3-',    'V11',     'V?',    'V1-',
   'V2-3',    'V0+',    'V6-',    'V5-',    'V2-',    'V8-',   'V10-',
    'V6+',  'V9-10',   'V12+',    'V5+',    'V7+']
Length: 47, dtype: str

In [ ]:
boulder_df.head(10)

,area_path,area_name,is_boulder_area,climb_name,grade_v,grade_font,is_bouldering
0,Colorado City Area > Rock Worship,Rock Worship,False,Apostacy of Warren Jeffs,NaN,NaN,None
1,Colorado City Area > Rock Worship,Rock Worship,False,Alt Tower Traverse,NaN,NaN,None
2,Colorado City Area > Rock Worship,Rock Worship,False,Tweeties Demise,NaN,NaN,None
3,Colorado City Area > Rock Worship,Rock Worship,False,C-Section,NaN,NaN,None
4,Colorado City Area > Rock Worship,Rock Worship,False,Polydigitorum,NaN,NaN,None
5,Colorado City Area > Cottonwood Mesa,Cottonwood Mesa,False,Wally's Route,NaN,NaN,None
6,Colorado City Area > Cottonwood Mesa,Cottonwood Mesa,False,Deception,NaN,NaN,None
7,Colorado City Area > Cottonwood Mesa,Cottonwood Mesa,False,Second-hander,NaN,NaN,None
8,Colorado Boulder,Colorado Boulder,False,Freebird,V2,NaN,True
9,Colorado Boulder,Colorado Boulder,False,Pan Am,V2,NaN,True


### Testing Database 


In [3]:
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    database="climbweather",
    user="postgres",
    password="1234"
)

cur = conn.cursor()

cur.exectute(
    """
    INSERT INTO areas (uuid, area_name, parent_id, lat, lng)
    VALUES (%s, %s, %s, %s, %s)
    RETURNING id;
    """, 
    ('test-uuid-4', 'Bishop', NULL, 33.99, -100.99)
)

new_id = cur.fetchone()[0]
print(f"Inserted area with the id: {new_id}")

conn.commit()
cur.close()
conn.close()

AttributeError: 'psycopg2.extensions.cursor' object has no attribute 'exectute'